# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** Optional stretch card (retired 2026-07-13; its core
now lives in ML-07). Kept because it changed a conclusion: one verdict here **overturned a finding I had
already written down** in `w02_ml_task_framing.ipynb`.

Every test gets a verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE**, with n beside every rate.

Continues from `w03_feature_leakage_check.ipynb`.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


Working dir: /content/Rayanflyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421


## 1. Distributions

**Look before deciding.** Three things in these distributions dictate how every later test must be read.

**1. The traffic columns are brutally heavy-tailed.** `impressions_90d` has median **731** but a max of
**517,715** — skew **11.4**. `clicks_90d` is skewed **18.3** (median **1**, max 4,178), `sessions_90d`
**12.1**. Consequences: means are meaningless here (I use medians and banded rates throughout), and any
raw-value threshold is effectively a threshold on the largest few clients' pages.

**2. The rate columns are mostly zero.** **13,212 of 30,000** pages have `ctr == 0` and **21,629** have
`engagement_rate == 0`. These are not "pages that convert badly" — they are pages with so little traffic
that the ratio has no content. Any test that averages a rate across the whole file is really measuring
how many tiny pages it contains.

**3. Freshness is bimodal and lopsided.** `days_since_last_update` has median **20** with 68% of pages in
the `0-30` tier, and only **174** pages in the `181+` tier. The classic "stale content" story barely
exists in this slice — worth knowing *before* building a rule that leans on staleness.

Read together, these say: **condition on volume before believing any rate signal**, and **report n
beside every bucket** because the interesting buckets are often tiny.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
TRAFFIC = ["impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d"]
print("Heavy tails - median vs max vs skew (means are useless here):")
tails = pd.DataFrame({
    "median": df[TRAFFIC].median(),
    "p90": df[TRAFFIC].quantile(0.90),
    "p99": df[TRAFFIC].quantile(0.99),
    "max": df[TRAFFIC].max(),
    "skew": df[TRAFFIC].skew(),
}).round(1)
print(tails.to_string())
print()

print("Rate columns are mostly exactly zero (no traffic -> no meaningful ratio):")
for col in ("ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"):
    zeros = (df[col] == 0).sum()
    print(f"  {col:18s} == 0 in {zeros:6,} rows ({zeros / len(df):5.1%}) | max {df[col].max():.1f}")
print()

print("Freshness is lopsided - the 'stale content' story barely exists here:")
print(df["freshness_tier"].value_counts(dropna=False).to_string())
print(f"  days_since_last_update: median {df['days_since_last_update'].median():.0f}, "
      f"max {df['days_since_last_update'].max()}")
print()
print("Position readings:")
print(df["position_tier"].value_counts().to_string())
print(f"  avg_position == 0 (no data, not rank 0): {(df['avg_position'] == 0).sum():,}")

Heavy tails - median vs max vs skew (means are useless here):
                 median      p90      p99     max  skew
impressions_90d   731.0  12136.4  73505.8  517715  11.4
clicks_90d          1.0     32.0    253.0    4178  18.3
sessions_90d        7.0     88.0    451.0    4345  12.1
pageviews_90d       8.0    116.0    648.0    5998  10.9

Rate columns are mostly exactly zero (no traffic -> no meaningful ratio):
  ctr                == 0 in 13,212 rows (44.0%) | max 100.0
  engagement_rate    == 0 in 21,629 rows (72.1%) | max 100.0
  scroll_rate        == 0 in 11,235 rows (37.5%) | max 300.0
  ai_traffic_pct     == 0 in 28,070 rows (93.6%) | max 300.0

Freshness is lopsided - the 'stale content' story barely exists here:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
  days_since_last_update: median 20, max 373

Position readings:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
  avg_position == 0 

## 2. Signal test #1 / #2 / #3 (verdict each)

Three signals a content team would plausibly act on. Each is tested against the label with n reported,
and each gets a verdict.

---

### Signal #1 — "Stale pages decline more." → **MIXED**

| `freshness_tier` | Declining rate | n |
|---|---|---|
| `0-30` | 0.511 | 20,480 |
| `31-90` | 0.589 | 175 |
| `91-180` | **0.611** | 9,171 |
| `181+` | **0.471** | 174 |

Directionally true up to 180 days: pages untouched for 3–6 months decline at **0.611** vs **0.511** for
freshly-updated ones — a real 10-point gap on large n. But the effect **reverses in the `181+` tier**
(0.471), which is *below* the base rate. With **n=174** I will not build a story on that reversal; the
honest statement is that the signal holds in the 90–180 day band and is unmeasurable beyond it.
**Verdict: MIXED.** Usable as one point in a score, not as a gate.

---

### Signal #2 — "Worse position means more decline." → **OPPOSITE (non-monotone)**

| `position_tier` | Declining rate | n |
|---|---|---|
| `top_3` | **0.241** | 2,321 |
| `page_1` | 0.570 | 11,814 |
| `striking` | **0.610** | 7,304 |
| `page_3_5` | 0.562 | 7,242 |
| `deep` | **0.344** | 1,319 |

The intuition is simply wrong. Decline peaks in the **middle** (`striking`, positions 11–20, at 0.610)
and is *lowest* at both extremes — `top_3` pages are the most stable in the portfolio (0.241, less than
half the base rate), and `deep` pages (position > 50) also decline less, because they have little traffic
left to lose. **Verdict: OPPOSITE** for the monotone version of the claim; the real shape is an inverted
U. This is why my ML-07 rule uses a *band* (`3 < position ≤ 50`) rather than a threshold.

---

### Signal #3 — "Pages with thin measurable coverage are decaying." → **CONFIRMED, with a reversal at the floor**

| `days_with_impressions` | Declining rate | n |
|---|---|---|
| 1–4 | **0.149** | 3,157 |
| 5–20 | 0.495 | 2,924 |
| 21–46 | 0.618 | 2,992 |
| 47–69 | **0.649** | 3,088 |
| 70–81 | 0.647 | 2,983 |
| 82–87 | 0.616 | 3,864 |
| 88 (the GSC cap) | 0.562 | 10,992 |

This is the strongest single signal in the file (AUC 0.579) and its shape is instructive: decline rate
rises steeply from **0.149** to **0.649**, then eases back at the 88-day cap. The floor reversal is not
noise — it is **structural**: a page needs prior-window impressions to be labelable as `down` at all, so
pages with almost no coverage are mostly `new`/`flat` instead. **Verdict: CONFIRMED** in the 20–87 band,
with the understanding that the bottom of the range is an artifact of the label definition, not evidence
that near-invisible pages are healthy.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
def audit(name: str, grouping, verdict: str, order=None) -> pd.DataFrame:
    """Label rate + n for each bucket, with the base rate and a verdict printed."""
    table = df.groupby(grouping, observed=True)["is_declining_label"].agg(
        declining_rate="mean", n="size").round(3)
    if order is not None:
        table = table.reindex(order)
    spread = table["declining_rate"].max() - table["declining_rate"].min()
    print(f"--- {name}")
    print(table.to_string())
    print(f"    base rate {BASE_RATE:.3f} | spread {spread:.3f} | VERDICT: {verdict}")
    print()
    return table


audit("Signal 1: stale pages decline more", "freshness_tier", "MIXED (holds to 180d, reverses at n=174)",
      order=["0-30", "31-90", "91-180", "181+"])

audit("Signal 2: worse position means more decline", "position_tier",
      "OPPOSITE - decline peaks in the middle, an inverted U",
      order=["top_3", "page_1", "striking", "page_3_5", "deep"])

coverage_bands = pd.cut(df["days_with_impressions"], [0, 4, 20, 46, 69, 81, 87, 90],
                        labels=["1-4", "5-20", "21-46", "47-69", "70-81", "82-87", "88+"])
audit("Signal 3: thin measurable coverage means decay", coverage_bands,
      "CONFIRMED in the 20-87 band (floor reversal is a label artifact)")

# The floor reversal, explained rather than asserted.
floor = df[df["days_with_impressions"] <= 4]
print("Why the 1-4 day band declines least - it is structural, not healthy:")
print(floor["trend_direction"].value_counts(normalize=True).round(3).to_string())
print(f"  {(floor['impressions_prev_30d'] == 0).mean():.1%} of that band has zero prior-window impressions,")
print("  so the label CANNOT be 'down' - they are 'new' or 'flat' by definition.")

--- Signal 1: stale pages decline more
                declining_rate      n
freshness_tier                       
0-30                     0.511  20480
31-90                    0.589    175
91-180                   0.611   9171
181+                     0.471    174
    base rate 0.542 | spread 0.140 | VERDICT: MIXED (holds to 180d, reverses at n=174)

--- Signal 2: worse position means more decline
               declining_rate      n
position_tier                       
top_3                   0.241   2321
page_1                  0.570  11814
striking                0.610   7304
page_3_5                0.562   7242
deep                    0.344   1319
    base rate 0.542 | spread 0.369 | VERDICT: OPPOSITE - decline peaks in the middle, an inverted U

--- Signal 3: thin measurable coverage means decay
                       declining_rate      n
days_with_impressions                       
1-4                             0.149   3157
5-20                            0.495   2924
21-46 

## 3. The flag-linked test

FlyRank's reference pipeline emits reason codes on its queue, two of which encode assumptions worth
testing directly (`outputs/model_report.md`): **`low_ctr_visible_page`** and
**`low_engagement_visible_page`**. Both assume: *a page with traffic but a poor conversion rate is a
refresh candidate.* Does the data support that?

### `low_ctr_visible_page` → **CONFIRMED (and it overturns my earlier finding)**

Among **visible** pages only (`impressions_90d ≥ 1000`, n=13,512), the label rate falls monotonically as
CTR rises:

| CTR band (×100 %) | Declining rate | n |
|---|---|---|
| ≤ 0.10 | **0.677** | 4,368 |
| 0.10–0.25 | 0.601 | 4,100 |
| 0.25–0.50 | 0.552 | 2,977 |
| 0.50–1.00 | 0.470 | 1,602 |
| > 1.00 | **0.460** | 465 |

A clean 22-point monotone gradient on large n. **The flag's assumption holds — conditional on
visibility.**

**This overturns what I wrote in ML-03.** In `w02_ml_task_framing.ipynb` I recorded CTR as a *negative
result*, because across the whole file in equal-sized quintiles the label rate moves only between 0.507
and 0.578. That reading was measured on the wrong population: **13,212 rows have `ctr == 0`**,
overwhelmingly tiny pages where CTR is a ratio over almost nothing. Without the visibility floor the same
banded test is both weaker and **non-monotone** (0.533 → 0.602 → 0.562 → 0.513 → 0.442, gradient 0.091
against 0.217 among visible pages) — muddy enough that quintiles read as noise. The signal was there; my
population obscured it. I have corrected the ML-03 notebook rather than leaving the two statements to contradict each
other, and the correction strengthens the case for a model: *CTR matters only in interaction with volume*,
which is exactly what a single threshold cannot express.

### `low_engagement_visible_page` → **FALSE (untestable as specified)**

The same test on `engagement_rate` collapses, because the variable barely varies:

| `engagement_rate` band | Declining rate | n |
|---|---|---|
| 0–25 | 0.593 | **13,381** |
| 25–50 | 0.687 | 115 |
| 50–75 | 1.000 | **3** |
| 75–100 | 0.846 | 13 |

**99.0% of visible pages sit in the bottom band**, and every other bucket is too small to carry a rate —
the "1.000" is three pages. `engagement_rate == 0` in **21,629** rows overall. **Verdict: FALSE as
specified** — not "engagement doesn't matter", but "this dataset cannot test it, and any flag firing on
engagement bands here is firing on noise". A headline like "pages with 50–75% engagement decline 100% of
the time" would be exactly the tiny-bucket claim to refuse.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
VISIBLE_FLOOR = 1000
visible = df[df["impressions_90d"] >= VISIBLE_FLOOR].copy()
print(f"visible pages (impressions_90d >= {VISIBLE_FLOOR:,}): {len(visible):,} "
      f"| declining rate {visible['is_declining_label'].mean():.3f}")
print()

# --- FLAG 1: low_ctr_visible_page -----------------------------------------
print("--- flag assumption: low CTR on a visible page => refresh candidate")
visible["ctr_band"] = pd.cut(visible["ctr"], [-0.01, 0.10, 0.25, 0.50, 1.00, 100])
ctr_table = visible.groupby("ctr_band", observed=True)["is_declining_label"].agg(
    declining_rate="mean", n="size").round(3)
print(ctr_table.to_string())
gradient = ctr_table["declining_rate"].iloc[0] - ctr_table["declining_rate"].iloc[-1]
monotone = ctr_table["declining_rate"].is_monotonic_decreasing
print(f"    monotone decreasing: {monotone} | gradient {gradient:.3f} | VERDICT: CONFIRMED (conditional on volume)")
print()

print("    ... and here is the same test WITHOUT the visibility floor - weaker and non-monotone:")
all_bands = pd.cut(df["ctr"], [-0.01, 0.10, 0.25, 0.50, 1.00, 100])
print(df.groupby(all_bands, observed=True)["is_declining_label"].agg(
    declining_rate="mean", n="size").round(3).to_string())
print(f"    {(df['ctr'] == 0).sum():,} rows have ctr == 0 - tiny pages flatten the whole-file view.")
print("    -> ML-03's 'CTR is flat' verdict was measured on the wrong population; corrected there.")
print()

# --- FLAG 2: low_engagement_visible_page ----------------------------------
print("--- flag assumption: low engagement on a visible page => refresh candidate")
visible["eng_band"] = pd.cut(visible["engagement_rate"], [-0.01, 25, 50, 75, 100])
eng_table = visible.groupby("eng_band", observed=True)["is_declining_label"].agg(
    declining_rate="mean", n="size").round(3)
print(eng_table.to_string())
biggest = eng_table["n"].max() / eng_table["n"].sum()
print(f"    {biggest:.1%} of visible pages sit in ONE band; smallest bucket n={eng_table['n'].min()}")
print(f"    engagement_rate == 0 in {(df['engagement_rate'] == 0).sum():,} rows overall")
print("    VERDICT: FALSE as specified - this dataset cannot test the assumption. Buckets too small.")
print()
tiny = eng_table[eng_table["n"] < 30]
print(f"    buckets I refuse to quote as rates (n < 30): {len(tiny)} of {len(eng_table)}")
print(tiny.to_string())

visible pages (impressions_90d >= 1,000): 13,512 | declining rate 0.594

--- flag assumption: low CTR on a visible page => refresh candidate
              declining_rate     n
ctr_band                          
(-0.01, 0.1]           0.677  4368
(0.1, 0.25]            0.601  4100
(0.25, 0.5]            0.552  2977
(0.5, 1.0]             0.470  1602
(1.0, 100.0]           0.460   465
    monotone decreasing: True | gradient 0.217 | VERDICT: CONFIRMED (conditional on volume)

    ... and here is the same test WITHOUT the visibility floor - weaker and non-monotone:
              declining_rate      n
ctr                                
(-0.01, 0.1]           0.533  16496
(0.1, 0.25]            0.602   5266
(0.25, 0.5]            0.562   4089
(0.5, 1.0]             0.513   2460
(1.0, 100.0]           0.442   1689
    13,212 rows have ctr == 0 - tiny pages flatten the whole-file view.
    -> ML-03's 'CTR is flat' verdict was measured on the wrong population; corrected there.

--- flag assum

## 4. What this means in practice

**For a content team, three takeaways.**

**Stop triaging on staleness alone, and stop triaging on position.** In this portfolio only 174 pages are
more than 180 days stale, so a "hasn't been touched in six months" rule has almost nothing to fire on —
and position points the opposite way from intuition: top-3 pages are the *most* stable (0.241 declining vs
a 0.542 base rate), while positions 11–20 are the most at risk (0.610). The pages worth an editor's
attention are the ones in the middle of the pack, not the ones ranking worst.

**Judge conversion signals only against pages that actually have traffic.** CTR looks like a dead signal
across the whole portfolio and a strong one among visible pages (0.677 declining at CTR ≤ 0.10 vs 0.460
above 1.0, n=13,512). Any dashboard that averages CTR over every published page is measuring its own
long tail. Set a visibility floor first — 1,000 impressions per 90 days worked here — then read the rate.

**Treat engagement-rate flags as unproven.** 21,629 of 30,000 pages have `engagement_rate == 0` and 99% of
visible pages fall in a single band, so any rule keyed on engagement bands in this data is firing on
noise, not on signal. That is a **negative result worth stating**: it tells the team where *not* to spend
its next measurement effort, and it flags an instrumentation gap rather than a content problem.

**Honest scope.** All of the above is **observed** in one trailing-90-day snapshot of 32 pseudonymized
clients, against a label that *defines* decline as a −20% impression change. None of it says that
refreshing a flagged page recovers traffic — that needs an experiment, and this data records no
intervention.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# The audit, as one summary table - the thing a content team would actually be handed.
verdicts = pd.DataFrame([
    {"signal": "stale pages decline more", "verdict": "MIXED",
     "effect": "0.511 (0-30d) -> 0.611 (91-180d), reverses to 0.471 at 181+",
     "n_note": "large n except 181+ (n=174)"},
    {"signal": "worse position = more decline", "verdict": "OPPOSITE",
     "effect": "inverted U: top_3 0.241, striking 0.610, deep 0.344",
     "n_note": "all buckets n>1,300"},
    {"signal": "thin measurable coverage = decay", "verdict": "CONFIRMED",
     "effect": "0.149 (1-4 days) -> 0.649 (47-69 days)",
     "n_note": "all buckets n>2,900"},
    {"signal": "low CTR on a VISIBLE page = decay", "verdict": "CONFIRMED",
     "effect": "0.677 (ctr<=0.10) -> 0.460 (ctr>1.0), monotone",
     "n_note": "n=13,512 visible pages"},
    {"signal": "low engagement on a visible page = decay", "verdict": "FALSE (untestable)",
     "effect": "99.0% of visible pages in one band; smallest bucket n=3",
     "n_note": "engagement_rate == 0 in 21,629 rows"},
])
print(f"SIGNAL AUDIT SUMMARY (base rate {BASE_RATE:.3f})")
print(verdicts.to_string(index=False))
print()
print("Score card: "
      f"{(verdicts['verdict'] == 'CONFIRMED').sum()} confirmed, "
      f"{(verdicts['verdict'] == 'MIXED').sum()} mixed, "
      f"{(verdicts['verdict'] == 'OPPOSITE').sum()} opposite, "
      f"{verdicts['verdict'].str.startswith('FALSE').sum()} false/untestable")
print()
print("Two of the five intuitions survived intact. That is the point of running the audit:")
print("one verdict (CTR) overturned a finding already written into ML-03, and one (engagement)")
print("identified an instrumentation gap rather than a content problem.")

SIGNAL AUDIT SUMMARY (base rate 0.542)
                                  signal            verdict                                                      effect                              n_note
                stale pages decline more              MIXED 0.511 (0-30d) -> 0.611 (91-180d), reverses to 0.471 at 181+         large n except 181+ (n=174)
           worse position = more decline           OPPOSITE         inverted U: top_3 0.241, striking 0.610, deep 0.344                 all buckets n>1,300
        thin measurable coverage = decay          CONFIRMED                      0.149 (1-4 days) -> 0.649 (47-69 days)                 all buckets n>2,900
       low CTR on a VISIBLE page = decay          CONFIRMED              0.677 (ctr<=0.10) -> 0.460 (ctr>1.0), monotone              n=13,512 visible pages
low engagement on a visible page = decay FALSE (untestable)     99.0% of visible pages in one band; smallest bucket n=3 engagement_rate == 0 in 21,629 rows

Score card: 2 confirmed,

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.